<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/ESG%20stock%20capital%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# esg_capital_stock_standalone.py
"""
Standalone program for ESG-Tobin's Q analysis using ESG Capital Stock models.
Focuses on creating and analyzing ESG capital stock with different depreciation rates.
"""
from google.colab import drive
drive.mount('/content/drive')

!pip install -q linearmodels pandas numpy matplotlib seaborn statsmodels
# esg_capital_stock_standalone.py
"""
Standalone program for ESG-Tobin's Q analysis using ESG Capital Stock models.
Focuses on creating and analyzing ESG capital stock with different depreciation rates.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

class ESGCapitalStockAnalysis:
    """
    Standalone class for running ESG Capital Stock models on Tobin's Q.
    Creates ESG capital stock variables with different depreciation rates.
    """

    def __init__(self, data_path, output_dir):
        """
        Initialize the ESG Capital Stock analysis.

        Parameters:
        -----------
        data_path : str
            Path to the Excel data file
        output_dir : str
            Directory to save output files
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.df = None
        self.capital_stock_data = {}
        self.results = {}

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Setup plotting style
        plt.style.use('seaborn-v0_8-whitegrid')
        sns.set_palette("husl")

    def load_and_prepare_data(self):
        """
        Load data and prepare basic transformations.
        """
        print("=" * 70)
        print("LOADING AND PREPARING DATA FOR ESG CAPITAL STOCK ANALYSIS")
        print("=" * 70)

        try:
            # Load data
            self.df = pd.read_excel(self.data_path)
            print(f"✓ Loaded data from: {self.data_path}")

            # Set multi-index
            if 'Firm' in self.df.columns and 'Year' in self.df.columns:
                self.df = self.df.set_index(['Firm', 'Year']).sort_index()
                print("✓ Set multi-index (Firm, Year)")
            else:
                print("⚠ Warning: 'Firm' and/or 'Year' columns not found")
                print(f"  Available columns: {list(self.df.columns)}")
                # Try to use first two columns as index
                self.df = self.df.set_index([self.df.columns[0], self.df.columns[1]]).sort_index()
                print(f"  Using {self.df.index.names} as index")

            # Rename columns if needed
            column_renames = {}
            if "Tobin's Q" in self.df.columns:
                column_renames["Tobin's Q"] = "Tobin_Q"

            # Check for ESG columns
            esg_columns = [col for col in ['E', 'S', 'G'] if col in self.df.columns]
            if len(esg_columns) == 3:
                print(f"✓ Found ESG columns: {esg_columns}")
            else:
                print(f"⚠ Missing ESG columns. Found: {esg_columns}")
                print(f"  All columns: {list(self.df.columns)}")

            # Apply renames if any
            if column_renames:
                self.df = self.df.rename(columns=column_renames)
                print(f"✓ Renamed columns: {column_renames}")

            # Create basic control variables
            if 'Total Assets' in self.df.columns:
                self.df['Size'] = np.log(self.df['Total Assets'].replace(0, np.nan))
                print("✓ Created Size variable (log of Total Assets)")
            else:
                print("⚠ Warning: 'Total Assets' column not found")

            if 'Total Liabilities' in self.df.columns and 'Total Assets' in self.df.columns:
                self.df['Leverage'] = self.df['Total Liabilities'] / self.df['Total Assets'].replace(0, np.nan)
                print("✓ Created Leverage variable")
            else:
                print("⚠ Warning: Could not create Leverage variable")

            # Create log transformation of Tobin's Q
            if 'Tobin_Q' in self.df.columns:
                self.df['Tobin_Q_log'] = np.log(self.df['Tobin_Q'].replace(0, np.nan) + 0.001)
                print("✓ Created Tobin_Q_log variable")
            else:
                print("⚠ Warning: 'Tobin_Q' column not found")

            # Create lagged ESG variables (t-1) if ESG columns exist
            for component in ['E', 'S', 'G']:
                if component in self.df.columns:
                    self.df[f'{component}_lag'] = self.df.groupby(level=0)[component].shift(1)
                    print(f"✓ Created {component}_lag variable")
                else:
                    print(f"⚠ Warning: {component} column not found, cannot create lag")

            print(f"\n✓ Loaded {len(self.df)} observations")
            print(f"✓ Number of firms: {self.df.index.get_level_values(0).nunique()}")
            print(f"✓ Years: {sorted(self.df.index.get_level_values(1).unique())}")

            # Print column summary
            print(f"\nAvailable columns ({len(self.df.columns)}):")
            for i, col in enumerate(sorted(self.df.columns)):
                if i < 20:  # Show first 20 columns
                    non_null = self.df[col].notna().sum()
                    print(f"  {col:<20} ({non_null} non-null)")

            return self.df

        except Exception as e:
            print(f"✗ Error loading data: {e}")
            import traceback
            traceback.print_exc()
            return None

    def create_sector_variable(self):
        """
        Create Consumer Staples sector variable.
        """
        print("\n" + "-" * 50)
        print("CREATING SECTOR VARIABLES")
        print("-" * 50)

        # Define Consumer Staples firms
        consumer_staples_firms = [
            'Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG',
            'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever', 'Haleon Plc'
        ]

        # Get unique firm names
        firm_names = self.df.index.get_level_values(0).unique()
        print(f"Found {len(firm_names)} unique firms")

        # Check which Consumer Staples firms are in the data
        found_cs_firms = [firm for firm in consumer_staples_firms if firm in firm_names]
        print(f"Found {len(found_cs_firms)} Consumer Staples firms in data: {found_cs_firms}")

        # Create sector variable
        sectors = []
        consumer_staples_dummy = []

        for firm in self.df.index.get_level_values(0):
            if firm in consumer_staples_firms:
                sectors.append('Consumer Staples')
                consumer_staples_dummy.append(1)
            else:
                sectors.append('Other')
                consumer_staples_dummy.append(0)

        self.df['Sector'] = sectors
        self.df['ConsumerStaples'] = consumer_staples_dummy

        print(f"\n✓ Created sector variables:")
        print(f"  Consumer Staples firms: {self.df[self.df['ConsumerStaples']==1].index.get_level_values(0).nunique()}")
        print(f"  Other sectors firms: {self.df[self.df['ConsumerStaples']==0].index.get_level_values(0).nunique()}")
        print(f"  Consumer Staples observations: {self.df['ConsumerStaples'].sum()}")

        return self.df

    def create_esg_capital_stock_debug(self, depreciation_rate=0.3, use_lagged_esg=True):
        """
        Debug version of capital stock creation with more detailed output.
        """
        print(f"\n{'='*70}")
        print(f"DEBUG: Creating ESG capital stock with δ={depreciation_rate:.2f}")
        print(f"Using {'lagged' if use_lagged_esg else 'current'} ESG")
        print(f"{'='*70}")

        # Determine which ESG variables to use
        if use_lagged_esg:
            esg_components = ['E_lag', 'S_lag', 'G_lag']
        else:
            esg_components = ['E', 'S', 'G']

        print(f"Looking for ESG components: {esg_components}")
        print(f"Available columns: {[c for c in self.df.columns if any(esg in c for esg in esg_components)]}")

        # Check which components are available
        available_components = [c for c in esg_components if c in self.df.columns]
        print(f"Available components: {available_components}")

        if not available_components:
            print("⚠ No ESG components found!")
            return self.df

        for component in available_components:
            print(f"\nProcessing {component}:")

            # Get base component name (without _lag if present)
            base_component = component.replace('_lag', '')
            col_name = f'{base_component}_capital_{int(depreciation_rate*100)}'

            print(f"  Creating column: {col_name}")

            # Process each firm separately
            capital_series = []
            firm_count = 0

            for firm in self.df.index.get_level_values(0).unique():
                firm_data = self.df.xs(firm, level=0).copy()

                if len(firm_data) > 0:
                    # Check if component exists for this firm
                    if component in firm_data.columns:
                        esg_series = firm_data[component].fillna(0).values

                        # Calculate capital stock
                        capital = np.zeros(len(esg_series))

                        if len(esg_series) > 0:
                            capital[0] = esg_series[0]  # Initial capital

                            for t in range(1, len(esg_series)):
                                capital[t] = (1 - depreciation_rate) * capital[t-1] + esg_series[t]

                        capital_series.extend(capital)
                        firm_count += 1
                    else:
                        # If component doesn't exist, add NaNs
                        capital_series.extend([np.nan] * len(firm_data))
                else:
                    # If no data for this firm, add NaNs
                    capital_series.extend([np.nan] * len(self.df.xs(firm, level=0)))

            # Add to dataframe
            self.df[col_name] = capital_series

            # Store metadata
            self.capital_stock_data[col_name] = {
                'component': component,
                'depreciation_rate': depreciation_rate,
                'use_lagged_esg': use_lagged_esg,
                'firms_processed': firm_count
            }

            # Print summary
            non_null = self.df[col_name].notna().sum()
            mean_val = self.df[col_name].mean()
            print(f"  ✓ Created {col_name}: {non_null} non-null values, mean={mean_val:.4f}")

        print(f"\n✓ Created ESG capital stock variables with {depreciation_rate*100:.0f}% depreciation")

        # Verify creation
        capital_cols = [f'{c}_capital_{int(depreciation_rate*100)}' for c in ['E', 'S', 'G']]
        created_cols = [col for col in capital_cols if col in self.df.columns]
        print(f"Created columns: {created_cols}")

        return self.df

    def create_esg_capital_stock(self, depreciation_rate=0.3, use_lagged_esg=True):
        """
        Create ESG capital stock variables with specified depreciation rate.
        """
        return self.create_esg_capital_stock_debug(depreciation_rate, use_lagged_esg)

    def create_multiple_depreciation_rates(self, rates=None):
        """
        Create ESG capital stock with multiple depreciation rates.

        Parameters:
        -----------
        rates : list
            List of depreciation rates to create (default: [0.1, 0.2, 0.3, 0.4, 0.5])
        """
        if rates is None:
            rates = [0.1, 0.2, 0.3, 0.4, 0.5]  # 10% to 50% depreciation

        print(f"\n{'='*70}")
        print(f"CREATING ESG CAPITAL STOCK WITH {len(rates)} DEPRECIATION RATES")
        print(f"Rates: {rates}")
        print(f"{'='*70}")

        for i, rate in enumerate(rates, 1):
            print(f"\n[{i}/{len(rates)}] Creating capital stock with δ={rate:.2f}")

            # Create with lagged ESG
            self.create_esg_capital_stock(depreciation_rate=rate, use_lagged_esg=True)

            # Also create with current ESG for comparison
            self.create_esg_capital_stock(depreciation_rate=rate, use_lagged_esg=False)

        print(f"\n✓ Created {len(rates) * 2} capital stock variables")

        # Print summary of created variables
        capital_cols = [col for col in self.df.columns if '_capital_' in col]
        print(f"\nCreated capital stock variables ({len(capital_cols)}):")
        for col in sorted(capital_cols):
            non_null = self.df[col].notna().sum()
            print(f"  {col:<25} ({non_null} non-null)")

        return self.df

    def run_basic_capital_stock_model(self, depreciation_rate=0.3, use_lagged_esg=True,
                                     y_var='Tobin_Q', sector='All'):
        """
        Simple, robust version of capital stock model.
        """
        print(f"\n{'='*70}")
        print(f"BASIC CAPITAL STOCK MODEL")
        print(f"Depreciation: {depreciation_rate*100:.0f}%, Lagged ESG: {use_lagged_esg}")
        print(f"Sector: {sector}, Y: {y_var}")
        print(f"{'='*70}")

        # Check if variables exist
        capital_vars = [f'{c}_capital_{int(depreciation_rate*100)}' for c in ['E', 'S', 'G']]

        print(f"Looking for capital variables: {capital_vars}")
        print(f"Available capital variables: {[col for col in capital_vars if col in self.df.columns]}")

        # Create variables if they don't exist
        missing_vars = [var for var in capital_vars if var not in self.df.columns]
        if missing_vars:
            print(f"Creating missing variables: {missing_vars}")
            self.create_esg_capital_stock(
                depreciation_rate=depreciation_rate,
                use_lagged_esg=use_lagged_esg
            )

        # Filter data by sector
        if sector == 'CS':
            if 'ConsumerStaples' not in self.df.columns:
                self.create_sector_variable()
            df_filtered = self.df[self.df['ConsumerStaples'] == 1].copy()
        else:
            df_filtered = self.df.copy()
            if 'ConsumerStaples' not in self.df.columns:
                self.create_sector_variable()

        # Define control variables
        control_vars = []
        for var in ['ROA', 'Size', 'Leverage']:
            if var in df_filtered.columns:
                control_vars.append(var)
            else:
                print(f"⚠ Control variable '{var}' not found")

        if sector == 'All':
            control_vars.append('ConsumerStaples')

        print(f"Using control variables: {control_vars}")

        # Clean data
        all_vars = capital_vars + control_vars + [y_var]

        # Check which variables are actually available
        available_vars = [var for var in all_vars if var in df_filtered.columns]
        missing_vars = [var for var in all_vars if var not in df_filtered.columns]

        print(f"Available variables: {available_vars}")
        if missing_vars:
            print(f"Missing variables: {missing_vars}")
            print("Trying to proceed with available variables...")
            all_vars = available_vars

        if y_var not in df_filtered.columns:
            print(f"✗ Dependent variable '{y_var}' not found!")
            print(f"Available Y variables: {[col for col in df_filtered.columns if 'Tobin' in col or 'Q' in col]}")
            return None

        df_clean = df_filtered.dropna(subset=all_vars)

        if len(df_clean) < 10:
            print(f"✗ Insufficient data: Only {len(df_clean)} observations")
            print(f"Available observations by variable:")
            for var in all_vars:
                non_null = df_filtered[var].notna().sum() if var in df_filtered.columns else 0
                print(f"  {var}: {non_null}")
            return None

        # Print data summary
        print(f"\nData Summary:")
        print(f"  Observations: {len(df_clean)}")
        print(f"  Firms: {df_clean.index.get_level_values(0).nunique()}")
        print(f"  Years: {sorted(df_clean.index.get_level_values(1).unique())}")

        if sector == 'All':
            cs_firms = df_clean[df_clean['ConsumerStaples']==1].index.get_level_values(0).nunique()
            other_firms = df_clean[df_clean['ConsumerStaples']==0].index.get_level_values(0).nunique()
            print(f"  Consumer Staples firms: {cs_firms}")
            print(f"  Other firms: {other_firms}")

        # Prepare data for regression
        y = df_clean[y_var]
        X_vars = [var for var in capital_vars + control_vars if var in df_clean.columns]
        X = df_clean[X_vars]
        X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

        print(f"\nRegression variables:")
        print(f"  Y: {y_var}")
        print(f"  X: {X_vars}")

        try:
            # Run regression
            model = PanelOLS(y, X, entity_effects=False, time_effects=False)
            results = model.fit(cov_type='robust')

            # Store results
            suffix = '_lag' if use_lagged_esg else ''
            model_key = f'Capital_δ{int(depreciation_rate*100)}{suffix}_{sector}_{y_var}'
            self.results[model_key] = {
                'results': results,
                'model_name': f'ESG Capital Stock (δ={depreciation_rate:.2f})',
                'depreciation_rate': depreciation_rate,
                'use_lagged_esg': use_lagged_esg,
                'y_var': y_var,
                'sector': sector,
                'nobs': results.nobs,
                'r2': results.rsquared,
                'capital_vars': capital_vars,
                'control_vars': control_vars
            }

            # Print results
            print(f"\n✓ Regression successful!")
            print(f"  R-squared: {results.rsquared:.4f}")

            if hasattr(results, 'f_statistic') and results.f_statistic:
                print(f"  F-statistic: {results.f_statistic.stat:.4f} (p={results.f_statistic.pval:.4f})")

            print("\nESG Capital Stock Coefficients:")
            for var in capital_vars:
                if var in results.params:
                    coeff = results.params[var]
                    pval = results.pvalues.get(var, 1.0)
                    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                    component = var.split('_')[0]
                    print(f"  {component} Capital: {coeff:.4f}{sig} (p={pval:.4f})")

            print("\nControl Variables:")
            for var in control_vars:
                if var in results.params:
                    coeff = results.params[var]
                    pval = results.pvalues.get(var, 1.0)
                    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                    print(f"  {var}: {coeff:.4f}{sig} (p={pval:.4f})")

            # Generate simple equation
            print(f"\n{'='*70}")
            print("REGRESSION EQUATION")
            print(f"{'='*70}")

            eq_parts = []
            if 'const' in results.params:
                const = results.params['const']
                eq_parts.append(f"{const:.4f}")

            for var in X_vars:
                if var in results.params:
                    coeff = results.params[var]
                    pval = results.pvalues.get(var, 1.0)

                    stars = ""
                    if pval < 0.01:
                        stars = "***"
                    elif pval < 0.05:
                        stars = "**"
                    elif pval < 0.10:
                        stars = "*"

                    if '_capital_' in var:
                        display_var = var.split('_')[0] + '_Capital'
                    else:
                        display_var = var

                    if coeff >= 0:
                        eq_parts.append(f"+ {abs(coeff):.4f}{stars}·{display_var}")
                    else:
                        eq_parts.append(f"- {abs(coeff):.4f}{stars}·{display_var}")

            equation = f"{y_var} = " + " ".join(eq_parts) + " + ε"
            print(equation)
            print(f"\nR² = {results.rsquared:.4f}, N = {results.nobs}")
            print(f"Depreciation δ = {depreciation_rate:.2f}")
            print("Significance: *** p<0.01, ** p<0.05, * p<0.10")

            return results

        except Exception as e:
            print(f"\n✗ Error running regression: {e}")
            import traceback
            traceback.print_exc()
            return None

    def run_capital_stock_model(self, depreciation_rate=0.3, use_lagged_esg=True,
                               y_var='Tobin_Q', sector='All'):
        """Alias for run_basic_capital_stock_model"""
        return self.run_basic_capital_stock_model(depreciation_rate, use_lagged_esg, y_var, sector)

    def run_simple_test(self):
        """
        Run a simple test to verify the capital stock creation works.
        """
        print("=" * 70)
        print("RUNNING SIMPLE TEST")
        print("=" * 70)

        # Test 1: Check data structure
        print("\n1. Checking data structure...")
        print(f"   Shape: {self.df.shape}")
        print(f"   Columns: {len(self.df.columns)}")
        print(f"   Index levels: {self.df.index.names}")

        # Test 2: Check for required columns
        print("\n2. Checking for required columns...")
        required_cols = ['E', 'S', 'G', 'Tobin_Q', 'ROA']
        for col in required_cols:
            if col in self.df.columns:
                non_null = self.df[col].notna().sum()
                print(f"   ✓ {col}: {non_null} non-null values")
            else:
                print(f"   ✗ {col}: NOT FOUND")

        # Test 3: Create capital stock with 30% depreciation
        print("\n3. Creating capital stock with 30% depreciation...")
        self.create_esg_capital_stock(depreciation_rate=0.3, use_lagged_esg=True)

        # Check if creation was successful
        capital_cols = [f'{c}_capital_30' for c in ['E', 'S', 'G']]
        created_cols = [col for col in capital_cols if col in self.df.columns]
        print(f"   Created columns: {created_cols}")

        for col in created_cols:
            non_null = self.df[col].notna().sum()
            mean_val = self.df[col].mean()
            print(f"   {col}: {non_null} non-null, mean={mean_val:.4f}")

        # Test 4: Run a simple regression
        print("\n4. Running simple regression test...")
        if len(created_cols) >= 2:  # Need at least 2 ESG components
            result = self.run_basic_capital_stock_model(
                depreciation_rate=0.3,
                use_lagged_esg=True,
                y_var='Tobin_Q',
                sector='All'
            )

            if result is not None:
                print("   ✓ Test passed!")
            else:
                print("   ✗ Test failed")
        else:
            print("   ✗ Not enough capital stock variables created")

        return len(created_cols) >= 2

    def run_depreciation_comparison(self, depreciation_rates=None):
        """
        Compare models with different depreciation rates.
        """
        if depreciation_rates is None:
            depreciation_rates = [0.1, 0.2, 0.3, 0.4, 0.5]

        print("=" * 70)
        print("DEPRECIATION RATE COMPARISON")
        print("=" * 70)

        comparison_data = []

        for rate in depreciation_rates:
            print(f"\nTesting δ = {rate:.1f}:")

            # Run model with lagged ESG
            result = self.run_basic_capital_stock_model(
                depreciation_rate=rate,
                use_lagged_esg=True,
                y_var='Tobin_Q',
                sector='All'
            )

            if result is not None:
                # Extract Social coefficient
                social_var = f'S_capital_{int(rate*100)}'
                if social_var in result.params:
                    social_coeff = result.params[social_var]
                    social_pval = result.pvalues.get(social_var, 1.0)
                else:
                    social_coeff = np.nan
                    social_pval = 1.0

                comparison_data.append({
                    'Depreciation_Rate': rate,
                    'R_Squared': result.rsquared,
                    'Social_Coefficient': social_coeff,
                    'Social_PValue': social_pval,
                    'Observations': result.nobs,
                    'Significant': social_pval < 0.10
                })

                print(f"  R² = {result.rsquared:.4f}, Social coeff = {social_coeff:.4f}")
            else:
                print(f"  Model failed")

        if comparison_data:
            # Create comparison DataFrame
            comp_df = pd.DataFrame(comparison_data)

            print(f"\n{'='*70}")
            print("COMPARISON RESULTS")
            print(f"{'='*70}")

            print(f"\n{'δ':<6} {'R²':<8} {'Social Coeff':<15} {'P-Value':<10} {'Sig':<6} {'N':<8}")
            print("-" * 60)

            for _, row in comp_df.iterrows():
                sig = "✓" if row['Significant'] else "✗"
                sig_level = ""
                pval = row['Social_PValue']
                if pval < 0.01:
                    sig_level = "***"
                elif pval < 0.05:
                    sig_level = "**"
                elif pval < 0.10:
                    sig_level = "*"

                print(f"{row['Depreciation_Rate']:<6.1f} {row['R_Squared']:<8.4f} "
                      f"{row['Social_Coefficient']:<15.4f} {pval:<10.4f} {sig}{sig_level:<5} {row['Observations']:<8}")

            # Find best model
            best_idx = comp_df['R_Squared'].idxmax()
            best_model = comp_df.loc[best_idx]

            print(f"\n{'='*70}")
            print("BEST MODEL:")
            print(f"{'='*70}")
            print(f"Depreciation rate: δ = {best_model['Depreciation_Rate']:.2f}")
            print(f"R-squared: {best_model['R_Squared']:.4f}")
            print(f"Social coefficient: {best_model['Social_Coefficient']:.4f}")
            print(f"Significance: {'Significant' if best_model['Significant'] else 'Not significant'}")

            return comp_df
        else:
            print("No comparison data available")
            return None

    def save_results(self):
        """
        Save results to CSV files.
        """
        print("\n" + "=" * 70)
        print("SAVING RESULTS")
        print("=" * 70)

        # Save capital stock variables summary
        if self.capital_stock_data:
            capital_summary = []
            for col_name, data_info in self.capital_stock_data.items():
                if col_name in self.df.columns:
                    non_null = self.df[col_name].notna().sum()
                    mean_val = self.df[col_name].mean()
                    std_val = self.df[col_name].std()

                    capital_summary.append({
                        'Variable': col_name,
                        'Component': data_info['component'],
                        'Depreciation_Rate': data_info['depreciation_rate'],
                        'Use_Lagged_ESG': data_info['use_lagged_esg'],
                        'Non_Null': non_null,
                        'Mean': mean_val,
                        'Std': std_val,
                        'Firms_Processed': data_info.get('firms_processed', 0)
                    })

            if capital_summary:
                capital_df = pd.DataFrame(capital_summary)
                capital_path = os.path.join(self.output_dir, 'capital_stock_variables.csv')
                capital_df.to_csv(capital_path, index=False)
                print(f"✓ Saved capital stock variables summary to: {capital_path}")

        # Save model results
        if self.results:
            results_summary = []

            for model_key, model_info in self.results.items():
                results = model_info['results']

                # Extract coefficients for ESG components
                for component in ['E', 'S', 'G']:
                    capital_var = f'{component}_capital_{int(model_info["depreciation_rate"]*100)}'

                    if capital_var in results.params:
                        coeff = results.params[capital_var]
                        pval = results.pvalues.get(capital_var, 1.0)

                        results_summary.append({
                            'Model_Key': model_key,
                            'Model_Name': model_info['model_name'],
                            'Depreciation_Rate': model_info['depreciation_rate'],
                            'Use_Lagged_ESG': model_info['use_lagged_esg'],
                            'Component': component,
                            'Y_Variable': model_info['y_var'],
                            'Sector': model_info['sector'],
                            'Observations': results.nobs,
                            'R_Squared': results.rsquared,
                            'Coefficient': coeff,
                            'P_Value': pval,
                            'Significant': pval < 0.10
                        })

            if results_summary:
                results_df = pd.DataFrame(results_summary)
                results_path = os.path.join(self.output_dir, 'capital_stock_results.csv')
                results_df.to_csv(results_path, index=False)
                print(f"✓ Saved model results to: {results_path}")

        # Save data sample with capital stock variables
        capital_cols = [col for col in self.df.columns if '_capital_' in col]
        if capital_cols:
            # Select a few important columns
            important_cols = ['Tobin_Q', 'Tobin_Q_log', 'ROA', 'Size', 'Leverage', 'ConsumerStaples'] + capital_cols
            available_cols = [col for col in important_cols if col in self.df.columns]

            if available_cols:
                sample_df = self.df[available_cols].reset_index()
                sample_path = os.path.join(self.output_dir, 'data_sample_with_capital.csv')
                sample_df.head(100).to_csv(sample_path, index=False)  # Save first 100 rows
                print(f"✓ Saved data sample to: {sample_path}")

        print(f"\n✓ All results saved to: {self.output_dir}")


def main():
    """
    Main function to run the ESG Capital Stock analysis.
    """
    # Configuration
    DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx'
    OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/esg_capital_stock_output'

    print("=" * 70)
    print("ESG CAPITAL STOCK ANALYSIS - DEBUG VERSION")
    print("=" * 70)

    # Initialize analysis
    analyzer = ESGCapitalStockAnalysis(DATA_PATH, OUTPUT_DIR)

    # Step 1: Load and prepare data
    print("\n" + "=" * 70)
    print("STEP 1: LOADING DATA")
    print("=" * 70)

    df = analyzer.load_and_prepare_data()

    if df is None:
        print("✗ Failed to load data. Exiting.")
        return

    # Step 2: Create sector variable
    print("\n" + "=" * 70)
    print("STEP 2: CREATING SECTOR VARIABLES")
    print("=" * 70)

    analyzer.create_sector_variable()

    # Step 3: Run simple test
    print("\n" + "=" * 70)
    print("STEP 3: RUNNING SIMPLE TEST")
    print("=" * 70)

    test_passed = analyzer.run_simple_test()

    if not test_passed:
        print("\n⚠ Simple test failed. Please check your data.")
        print("Trying alternative approach...")

        # Try with only 2 components if available
        esg_components = [c for c in ['E', 'S', 'G'] if c in analyzer.df.columns]
        if len(esg_components) >= 2:
            print(f"\nFound {len(esg_components)} ESG components: {esg_components}")
            print("Will proceed with available components.")
        else:
            print("✗ Not enough ESG components. Exiting.")
            return

    # Step 4: Run depreciation rate comparison
    print("\n" + "=" * 70)
    print("STEP 4: DEPRECIATION RATE COMPARISON")
    print("=" * 70)

    depreciation_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
    comparison_results = analyzer.run_depreciation_comparison(depreciation_rates)

    # Step 5: Run models for Consumer Staples
    print("\n" + "=" * 70)
    print("STEP 5: CONSUMER STAPLES ANALYSIS")
    print("=" * 70)

    # Use best depreciation rate from comparison
    best_rate = 0.3  # Default
    if comparison_results is not None:
        best_idx = comparison_results['R_Squared'].idxmax()
        best_rate = comparison_results.loc[best_idx, 'Depreciation_Rate']
        print(f"Using best depreciation rate: δ = {best_rate:.2f}")

    # Run model for Consumer Staples
    cs_result = analyzer.run_basic_capital_stock_model(
        depreciation_rate=best_rate,
        use_lagged_esg=True,
        y_var='Tobin_Q',
        sector='CS'
    )

    # Step 6: Save results
    print("\n" + "=" * 70)
    print("STEP 6: SAVING RESULTS")
    print("=" * 70)

    analyzer.save_results()

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)

    # Summary
    print("\nSUMMARY:")
    print(f"1. ✓ Data loaded: {len(analyzer.df)} observations")
    print(f"2. ✓ Capital stock variables created: {len([c for c in analyzer.df.columns if '_capital_' in c])}")
    print(f"3. ✓ Models run: {len(analyzer.results)}")

    if analyzer.results:
        # Show best model
        best_model_key = None
        best_r2 = -1

        for model_key, model_info in analyzer.results.items():
            if model_info['r2'] > best_r2:
                best_r2 = model_info['r2']
                best_model_key = model_key

        if best_model_key:
            best_model = analyzer.results[best_model_key]
            print(f"\nBEST MODEL:")
            print(f"  Name: {best_model['model_name']}")
            print(f"  Sector: {best_model['sector']}")
            print(f"  Y: {best_model['y_var']}")
            print(f"  R²: {best_model['r2']:.4f}")
            print(f"  N: {best_model['nobs']}")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ESG CAPITAL STOCK ANALYSIS - DEBUG VERSION

STEP 1: LOADING DATA
LOADING AND PREPARING DATA FOR ESG CAPITAL STOCK ANALYSIS
✓ Loaded data from: /content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx
✓ Set multi-index (Firm, Year)
✓ Found ESG columns: ['E', 'S', 'G']
✓ Created Size variable (log of Total Assets)
✓ Created Leverage variable
✓ Created Tobin_Q_log variable
✓ Created E_lag variable
✓ Created S_lag variable
✓ Created G_lag variable

✓ Loaded 388 observations
✓ Number of firms: 40
✓ Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Available columns (19):
  E                    (388 non-null)
  ESG                  (388 non-null)
  E_lag                (348 non-null)
  G                    (388 non-null)
  G_lag                (348 non-null)
  Leverage             (388 non-null)
  Market Value of Equity (388 non-null)
  Ne